# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR² dataset using the `mlcroissant` library. All dataset components are referenced by their `@id` fields, ensuring schema-level clarity, reproducibility, and provenance throughout.

### Dataset Source
The dataset is distributed as a Croissant schema at the following URL:
https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
We load the dataset metadata and record set structure using the `mlcroissant` library. This enables access to dataset details, record sets, fields, and their unique `@id` identifiers.


In [ ]:
import mlcroissant as mlc
import pandas as pd

# Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load dataset. The metadata attribute provides access to the dataset's metadata object.
ds = mlc.Dataset(croissant_url)
metadata = ds.metadata

print(f"Dataset title: {metadata.name}\nDescription: {metadata.description}\n")

## 2. Data Overview
We examine the available record sets in the dataset, and for each, list their fields (`@id` and name), so you can reference them directly in further steps.

> **Note:** All entities are referenced by their `@id` fields.

In [ ]:
# List available record sets, fields, and fields' @id and name
print("=== Record Sets ===")
all_record_sets = []

for record_set in metadata.record_sets:
    print(f"Record Set: {record_set.name}  (@id: {record_set.id})")
    all_record_sets.append(record_set.id)
    print("  Fields:")
    for field in record_set.fields:
        col_ids = [c.id for c in field.columns] if field.columns else []
        print(f"    - {field.name}  (@id: {field.id})  Columns: {col_ids}")
    print()

# Show a sample record (if record set not empty)
print("\n=== Sample Records from Each Record Set ===")
for record_set_id in all_record_sets:
    print(f"\nFirst record from {record_set_id}:")
    try:
        records_iter = ds.records(record_set=record_set_id)
        record = next(records_iter)
        print(record)
    except StopIteration:
        print("No records found.")
    except Exception as e:
        print(f"Error: {e}")

## 3. Data Extraction
Let's extract data from each record set using their `@id` fields. We'll load each as a DataFrame for analysis. For demonstration, we'll print the fields of one record set as an example.

In [ ]:
# List of record set @ids, found in the previous step
record_sets = all_record_sets  # e.g., ['cr:main', ...]

dataframes = {}

for record_set_id in record_sets:
    records = list(ds.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Columns for {record_set_id}: {df.columns.tolist()}")

# If there's a main record set (by convention e.g. 'cr:main' or 'cr:table_1'), preview its data
main_rs = record_sets[0] if record_sets else None
if main_rs is not None:
    print(f"\nPreview of {main_rs}:")
    display(dataframes[main_rs].head())

## 4. Exploratory Data Analysis (EDA)
We demonstrate how to filter, normalize, and group records by referencing column `@id`s. Please adjust `numeric_field_id` and `group_field_id` below according to the ID printed in the overview.

In [ ]:
# Specify the main record set and field ids by their @id
# Replace the below field IDs with the actual numeric and group field ids from step 2 output.
# Example: numeric_field_id = 'cr:abundance'   group_field_id = 'cr:sample_id' (use actual @ids)
main_record_set_id = main_rs

# User-provided: adjust to match a real numeric field @id
numeric_field_id = None  # e.g. 'cr:abundance' or 'cr:peptide_count'
group_field_id = None    # e.g. 'cr:sample_id' or 'cr:condition'

# Identify a numeric field in the main record set
for col in dataframes[main_record_set_id].columns:
    # Heuristically choose first numeric-looking column
    if dataframes[main_record_set_id][col].dtype in [int, float, 'float64', 'int64']:
        numeric_field_id = col
        break

# Identify a group field (non-numeric)
for col in dataframes[main_record_set_id].columns:
    if (col != numeric_field_id) and (dataframes[main_record_set_id][col].dtype == object):
        group_field_id = col
        break

print(f"Using numeric_field_id: {numeric_field_id}")
print(f"Using group_field_id: {group_field_id}")

# Remove outliers (values > mean + 3*std) and normalize
df = dataframes[main_record_set_id]
if numeric_field_id is not None:
    mu = df[numeric_field_id].mean()
    sigma = df[numeric_field_id].std()
    filtered_df = df[df[numeric_field_id] < mu + 3*sigma].copy()
    print(f"Filtered {numeric_field_id} < mean + 3*std (mean={mu:.2f}, std={sigma:.2f})")

    # Normalize
    filtered_df[f"{numeric_field_id}_normalized"] = (
        filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
    ) / filtered_df[numeric_field_id].std()
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    # Group by a categorical field (if available)
    if group_field_id is not None:
        grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nMean {numeric_field_id} by {group_field_id}:")
        print(grouped.head())

## 5. Visualization
Let's visualize the distribution of the selected numeric field and how it varies by group (if grouping field is present).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None:
    plt.figure(figsize=(7, 4))
    sns.histplot(filtered_df[numeric_field_id], kde=True, color="teal")
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.tight_layout()
    plt.show()

if (numeric_field_id is not None) and (group_field_id is not None):
    plt.figure(figsize=(10,5))
    sns.boxplot(x=group_field_id, y=numeric_field_id, data=filtered_df)
    plt.title(f"{numeric_field_id} by {group_field_id}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
* The FAIR² dataset was loaded and explored using schema-level identifiers via `mlcroissant`.
* Record sets, fields, and columns were referenced and extracted by their `@id`.
* We filtered, normalized, grouped, and visualized data using reproducible code based on schema structures.

For further analysis, refer to the Croissant schema for more detailed field descriptions, or extend filtering/grouping logic by other `@id` values as needed.

For more details on the `mlcroissant` library, see [mlcroissant documentation](https://mlcommons.github.io/croissant/api/).
